In [ ]:
!python3 -m pip install ipykernel -U --user --force-reinstall

In [ ]:
!pip install openai python-dotenv pillow opencv-python numpy

In [ ]:
import os
from dotenv import load_dotenv
import openai
import base64
from PIL import Image
import csv
import time
import json
from dotenv import load_dotenv

In [ ]:
load_dotenv()

api_key = os.getenv("API_KEY")
print("API Key loaded:", "✅" if api_key else "❌ Not Found")

In [ ]:
# I try to use vision models - specific for image but not available in the free tier, so I look up the models
models = openai.models.list()
for model in models.data:
    print(model.id)

In [ ]:
# 🔐 Load API key
openai.api_key = api_key

# 📂 Paths
image_folder = "./images"
output_json = "store_sign_results.json"

# 🔢 How many new images to process in this run
BATCH_LIMIT = 100  # ← Change this to 500, 1000, etc. as needed

# ========================
# HELPER FUNCTIONS
# ========================

# 🔠 Convert filename to ID (without extension)
def get_id(filename):
    return os.path.splitext(filename)[0]

# 📸 Convert image to base64
def encode_image(path):
    with open(path, "rb") as img:
        return base64.b64encode(img.read()).decode("utf-8")

# 🧠 Ask GPT-4o to analyze the image
def analyze_image(image_path):
    base64_img = encode_image(image_path)

    prompt = (
        "Please analyze this image and respond in **JSON format only** with these fields:\n"
        "{\n"
        "  \"has_sign\": \"Yes or No\",\n"
        "  \"sign_color\": \"e.g. Red/White\",\n"
        "  \"font_style\": \"Serif, Sans-serif, Script, Decorative, etc.\",\n"
        "}\n"
        "Only return the JSON object — no explanation or commentary."
    )

    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_img}"}
                    }
                ]
            }
        ],
        max_tokens=500,
        temperature=0.3
    )

    return response.choices[0].message.content.strip()

# ========================
# MAIN SCRIPT
# ========================

# 🔁 Load existing results
results = []
processed_ids = set()
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        results = json.load(f)
        processed_ids = {item["id"] for item in results}

# 🖼️ Get list of unprocessed images
image_files = sorted([f for f in os.listdir(image_folder) if f.lower().endswith((".jpg", ".jpeg"))])
to_process = [f for f in image_files if get_id(f) not in processed_ids][:BATCH_LIMIT]

print(f"\n🚀 Starting batch: Processing {len(to_process)} new image(s)...\n")

# 🧠 Analyze each image
for idx, filename in enumerate(to_process, 1):
    img_path = os.path.join(image_folder, filename)
    img_id = get_id(filename)

    try:
        print(f"🔍 [{idx}/{len(to_process)}] Analyzing {filename}...")
        raw_response = analyze_image(img_path)

        # Parse the JSON response from GPT
        # Clean markdown code block if present
        if raw_response.startswith("```json") or raw_response.startswith("```"):
            raw_response = raw_response.strip("```json").strip("```").strip()

        parsed = json.loads(raw_response)
        parsed["id"] = img_id
        results.append(parsed)

        # 💾 Save after every successful result
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        print(f"✅ Saved result for {filename}")

    except json.JSONDecodeError:
        print(f"❌ Could not parse JSON for {filename}. Raw response:\n{raw_response}\n")
        continue
    except openai.error.RateLimitError:
        print("⚠️ Rate limit hit. Sleeping 60 seconds...")
        time.sleep(60)
        continue
    except Exception as e:
        print(f"⚠️ Error processing {filename}: {e}")
        continue

    if idx % 50 == 0:
        print("⏸️ Pausing for 10 seconds to avoid overloading the API...")
        time.sleep(10)

print("\n🎉 Done! Results saved to store_sign_results.json\n")
